# 02805 Social graphs and interactions, Fall 20205 - Assignment 2

#### Søren Stange, s204229 
#### Freja Tusindfryd Dollas, s204248

Coding prerequisites:

In [23]:
#import all necessary libraries and modules
import networkx as nx 
import matplotlib.pyplot as plt
import numpy as np
import pickle, requests, os, re
import nltk
import string
import math
from collections import Counter
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

In [24]:
# Download the data from github if not already downloaded
filename = "assignment_1_rock_bands.pkl"

url = "https://raw.githubusercontent.com/sorenstange/02805_Social_graphs_and_interactions/main/assignment_1_rock_bands.pkl"

# Step 1: Check if file exists
if not os.path.exists(filename):
    # Download the file if it does not exist
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print("File downloaded successfully.")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")
else:
    print("File already exists. Skipping download.")

# Step 2: Load the .pickle file
with open(filename, "rb") as f:
    wikipedia_data = pickle.load(f)

File already exists. Skipping download.


In [25]:
# Declare helper functions
def clean_links(links):
    arr = []
    for l in links:
        match = re.search(r'|', l)
        if match:
            splits = l.split(r'|')
            arr.append(splits[0].replace(' ', '_'))
        else:
            arr.append(l.replace(' ', '_'))
    return arr

def analyze_node(data_point):
    Node = data_point['page_name']
    text = data_point['content']
    words = len(text.split())
    links_to = re.findall(r'\[\[([^\]]+)\]\]', text)
    links_to = clean_links(links_to)
    return Node, links_to, words

def create_network(data):
    G = nx.DiGraph()
    Nodes = list(set([data_point['page_name'] for data_point in data]))
    Nodes.remove('AllMusic')
    for data_point in data:
        Node, links_to, words = analyze_node(data_point)
        if Node == 'AllMusic':
            continue
        G.add_node(Node, words=words)
        for links_to_node in links_to:
            if links_to_node in Nodes:
                G.add_edge(Node, links_to_node)
    
    # Extract the largest component
    components = nx.weakly_connected_components(G)
    largest = max(components, key=len)
    G = G.subgraph(largest).copy()
    return G

In [26]:
# Create the directed graph
G = create_network(wikipedia_data)

In [27]:
# Extracting genres - 
# NB!!!!! this code is done very much with the help of chatGPT as stated is allowed in the weekly assignment. 
import re
from collections import Counter
import matplotlib.pyplot as plt

# --- Cleaning functions (same as before) ---

def clean_wiki_markup(text):
    if not text:
        return ""
    text = re.sub(r'<!--.*?-->', '', text, flags=re.S)
    text = re.sub(r'\{\{\s*flatlist\s*\|\s*([^}]*)\}\}', lambda m: re.sub(r'\*\s*', ', ', m.group(1)), text, flags=re.I)
    text = re.sub(r'\{\{nowrap\|([^}]*)\}\}', r'\1', text, flags=re.I)
    text = re.sub(r'\{\{[^\}]*\}\}', '', text)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\[https?:\/\/[^\s\]]+\s*([^\]]*)\]', r'\1', text)
    text = re.sub(r'https?:\/\/\S+', '', text)
    text = re.sub(r'url\s*=\s*\S+', '', text)
    text = re.sub(r'www\.[^\s]+', '', text)
    text = re.sub(r"''+", '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace('\n', ',').replace('•', ',').replace(';', ',')
    text = re.sub(r'[,/\\|]+', ',', text)
    text = re.sub(r'\s+', ' ', text).strip(' ,')
    return text

def normalize_genre_name(g):
    g = g.lower()
    g = g.replace("rock n roll", "rock and roll")
    g = g.replace("rock'n'roll", "rock and roll")
    g = g.replace("rock ’n’ roll", "rock and roll")
    g = g.replace("psychedelic-rock", "psychedelic rock")
    g = re.sub(r'\s*&\s*', ' and ', g)
    return g.strip()

def split_genres(text):
    if not text:
        return []
    text = re.sub(r'[\n;/]', ',', text)
    text = re.sub(r'\s*\*\s*', ',', text)
    text = re.sub(r'[|]', ',', text)
    genres = [g.strip().lower() for g in text.split(',') if g.strip()]
    blacklist = {
        'artist', 'website', 'label', 'albums', 'musician', 'url=', 
        'www.allmusic.com', 'discogs', 'music', 'genre', '}}', '{{', 
        'nowrap', 'flatlist'
    }
    return [g for g in genres if g not in blacklist and len(g) > 1]

def extract_genre_field(text):
    lines = text.splitlines()
    capture = False
    genre_lines = []
    for line in lines:
        if not capture:
            if re.match(r'\|\s*genre\s*=', line, flags=re.I):
                line = re.sub(r'\|\s*genre\s*=\s*', '', line, flags=re.I)
                genre_lines.append(line)
                capture = True
        else:
            if re.match(r'\|\s*\w+\s*=', line):
                break
            genre_lines.append(line)
    return '\n'.join(genre_lines).strip()

# --- Extract genres from wikipedia_data ---

genres_dict = {}

for entry in wikipedia_data:
    artist_name = entry['page_name']
    text = entry['content']

    raw = extract_genre_field(text)
    if raw:
        cleaned = clean_wiki_markup(raw)
        genres = [normalize_genre_name(g) for g in split_genres(cleaned)]
        if genres:
            genres_dict[artist_name] = genres

# --- Stats ---


num_with_genres = len(genres_dict)
total_genres = sum(len(v) for v in genres_dict.values())
distinct_genres = set(g for gs in genres_dict.values() for g in gs)


# --- Top genres ---

all_genres_flat = [g for gs in genres_dict.values() for g in gs]
genre_counts = Counter(all_genres_flat)
top15 = genre_counts.most_common(15)



## Part 1: Analyze the network


The questions in this part are based on Lecture 5.

- Present an analysis/description of the network of bands/artists using tools from Lecture 5. Imagine that you have been tasked with presenting the important facts about the network to an audience who knows about network science, but doesn't know about this particular network.
    * It's OK to also use basic concepts like degree distributions (even though they're from week 4) in your analysis. That way you can make the analysis a standalone, coherent thing.
    * I would like you to include concepts like centrality and assortativity in your analysis.
    * Use a network backbone in your analysis.
    * In addition to standard distribution plots (e.g. degree distributions, etc), your analysis should also include at least one network visualization (but it doesn't have to display the entire network, you can also visualize a network backbone).
    * **Note**: As I write above, an important part of the exercise consists is selecting the right elements of the lecture to create a meaningful analysis. So don't solve this part by going exhaustive and just calculating everything you can think of in one massive analysis. Try to focus on using what you've learned to characterize the network.

Jeg tænker at vi skal beskrive: 

- degree distribution + plot both in and out
- centrality (degree, betweenness, eigenvector - maybe scatterplots to compare? )
- assortativity (degree, length of content)
- average shortest path compared to random graph
- network backbone 
- visualization of network (e.g. network backbone)


## Part 2: Genres and communities and plotting

The questions below are based on Lecture 7, part 2.





##### Question: Write about genres and modularity.
**Answer**: The modularity of networks is a measure of how well a certain partitioning capture the underlying community structure. In this case we derive the genre of each band from the info box on their respective pages and use this information to partition the network. Then we can use the modularity as a measure of how well the partioning actually capture the communities of the network. Modularity is between 1 and 0, and a high modularity implies that partitioning captures the underlying community while a modularity closer to 0 indicates that the partitioning is quite bad. In this case it is a fair assumption that the genres of the rockbands can be an effective measure to capture some of the structure of the communities. 

##### Question: Detect the communities, discuss the value of modularity in comparison to the genres.

**Answer**:


In [28]:
#Firstly the modularity with respect to genre is calculated: 


def calculate_modularity(G, partitions):
    M = 0
    L = len(G.edges())
    if isinstance(partitions, dict):
        for key, nodes in partitions.items():
            subG = G.subgraph(nodes)
            Lc = len(subG.edges())
            kc = sum([deg for _, deg in subG.degree()])
            M += Lc / L - (kc/(2*L))**2
    elif isinstance(partitions, list):
        for nodes in partitions:
            nodes = list(nodes)
            subG = G.subgraph(nodes)
            Lc = len(subG.edges())
            kc = sum([deg for _, deg in subG.degree()])
            M += Lc / L - (kc/(2*L))**2
    
    return M

# Keep only nodes with genres
nodes_with_genres = [n for n in G.nodes if n in genres_dict]

# GH is undirected version
GH = G.subgraph(nodes_with_genres).to_undirected()

# Partition by first genre
partition_first = {}
for node in GH.nodes:
    partition_first[node] = genres_dict[node][0]  # first genre

communities_first = {}
for node, com in partition_first.items():
    communities_first.setdefault(com, set()).add(node)

communities_first = list(communities_first.values())


Q_first = calculate_modularity(GH, communities_first)
print("Modularity (first genre):", Q_first)

import networkx as nx
import random


# Partition by first non-rock genre
partition_non_rock = {}
for node in GH.nodes:
    non_rock = next((g for g in genres_dict[node] if g != "rock"), genres_dict[node][0])
    partition_non_rock[node] = non_rock

communities_non_rock = {}
for node, com in partition_non_rock.items():
    communities_non_rock.setdefault(com, set()).add(node)
communities_non_rock = list(communities_non_rock.values())


Q_non_rock = calculate_modularity(GH, communities_non_rock)
print("Modularity (first non-rock genre):", Q_non_rock)


# Partition by random genre from the list
partition_random = {node: random.choice(genres_dict[node]) for node in GH.nodes}

communities_random = {}
for node, com in partition_random.items():
    communities_random.setdefault(com, set()).add(node)
communities_random = list(communities_random.values())

Q_random = calculate_modularity(GH, communities_random)
print("Modularity (random genre assignment):", Q_random)


Modularity (first genre): 0.13933172289323892
Modularity (first non-rock genre): 0.13429411819149428
Modularity (random genre assignment): 0.052457698181313385


In [ ]:
#Secondly the modularity as calculated with the Louvain algorithm is dertemined: 
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity

# Compute Louvain communities
communities_louvain = louvain_communities(GH, seed=42)

# Compute modularity
Q_louvain = modularity(GH, communities_louvain)

# Print results
print(f"Detected {len(communities_louvain)} communities")
print(f"Modularity (Louvain): {Q_louvain:.4f}")


Detected 6 communities
Modularity (Louvain): 0.3408


The modularity values for partitioning the network by the first genre, the first non-rock genre, and randomly are 0.139, 0.134, and 0.0754, respectively. Compared to the modularity of 0.3408 obtained using the Louvain algorithm, these values are relatively low, indicating that genre alone is not an effective way of capturing the network’s community structure. However, genre-based partitioning still performs significantly better than a random assignment, which makes sense, since genre does contain some information about connections in the network.

A reason genre may be a poor measure is that many artists have multiple genre labels, and using only the first genre is somewhat arbitrary. Additionally, some genres are very broad, which can obscure community structures. The Louvain algorithm identifies six communities, a much smaller number than the total number of genres, suggesting that the true communities are defined more by the network’s connections than by genre labels. 

Overall these results highligt that while general information, like genre, can provide some insight into the community structure of the netwokr, the community detection is much more efficient. 



##### Question: Calculate the matrix and discuss your findings.
**Answer**:

In [29]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Most common genres: 
all_genres = [g for node in GH.nodes for g in genres_dict[node]]  # all genres for nodes in GH
top_genres = pd.Series(all_genres).value_counts().head(7).index.tolist()

#Determine community of genre
node_to_community = {}
for idx, com in enumerate(communities_louvain):
    for node in com:
        node_to_community[node] = idx

# Create confusion matrix
conf_matrix = pd.DataFrame(
    0,
    index=top_genres,
    columns=range(len(communities_louvain)) 
)

for node in GH.nodes:
    if node not in node_to_community:
        continue
    community = node_to_community[node]
    for genre in genres_dict[node]:
        if genre in top_genres:
            conf_matrix.loc[genre, community] += 1


print("D: Confusion Matrix (Genres vs Louvain Communities):")
print(conf_matrix)



D: Confusion Matrix (Genres vs Louvain Communities):
                    0   1   2   3  4   5
alternative rock   23   5  28  52  0   6
hard rock          29  45  13   2  0  10
pop rock            6  12   3  30  0  19
alternative metal  21  14  14   2  0   0
heavy metal         3  33   3   0  0   1
rock                1   5   1   2  0  30
post-grunge        28   2   6   2  0   0


From the confusion matrix above (6 communities and the 7 most common genres), it can be seen that for the genres alternative rock, heavy metal, rock, and post-grunge, there is a noticeably higher number of nodes in one community compared to others. However, all genres have nodes distributed across multiple communities (at least 4 communities for all genres), suggesting that genre alone does not capture the network’s partitioning. Additionally, each community contains multiple genres (except for community 5, which has no nodes from the chosen genres), further showing that community structure is not defined solely by genre. Overall, this suggests that while genre provides some information, it is a poor measure for partitioning the network into communities. This is consistent with the Louvain modularity results as found earlier, where partitioning into communities by the Louvan-algorithm had significantly higher modularity than any genre-based partitioning.

##### Question: Plot the communities and comment on your results.


## Part 3: TF-IDF to understand genres and communities

The questions below are based on Lecture 7, part 2, 4, 5, 6 (and a little bit on part 3).




##### Question: Explain the concept of TF-IDF in your own words and how it can help you understand the genres and communities.

**Answer**: TF-IDF is a statistical measure which can be used to assess the importance a specific word has in a text or in a collection of texts (corpus). It is the product of two statistics: *term frequency* (TF) and the *inverse document frequency*. TF measures how often a word appears in a single document, wheras IDF measures how rare a word is across a corpus. Thus when a word as a high TF-IDF in a document, it tells us that this word is frequent in the document, but rare in the other documents of the corpus. We can thus use TF-IDF as a measure to extract all the important keywords of a text, that is all the words which characterizes this text.

With this in mind, we can use TF-IDF to extract all the important keywords of the different genres or structural communities from the wikipedia pages, and then analyse the words which are found. This can be especially helpfull if we for instance use an algorithm for finding structural communities such as the Louvain-algorithm, to get more insight in what the community found actually represents. In the context of genre based communities, it can help us understand how the genres are represented in language.

##### Question: Calculate and visualize TF-IDF for the genres and communities.

**Answer**:

In [ ]:
# Declare helper functions
def create_corpus(wikipedia_data, partition, filter_n = 5):
    corpus = {}
    if isinstance(partition, dict):
        for key, nodes in partition.items():
            corpus[key] = ''
            for node in nodes:
                for data in wikipedia_data:
                    if data['page_name'] == node:
                        corpus[key] += data['content']

    elif isinstance(partition, list):
        for i, nodes in enumerate(partition):
            key = f'Community {i+1}'
            corpus[key] = ''
            for node in nodes:
                for data in wikipedia_data:
                    if data['page_name'] == node:
                        corpus[key] += data['content'] 

    lemmatizer = WordNetLemmatizer()
    for key, item in corpus.items():
        tokens = word_tokenize(item)
        tokens = [word for word in tokens if word.isalnum()]
        tokens = [word.lower() for word in tokens]
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
        word_counts = Counter(tokens)
        tokens = [word for word in tokens if word_counts[word] >= filter_n]
        token_counts = {word: count for word, count in word_counts.items() if count >= filter_n}
        token_counts = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)
        corpus[key] = token_counts
    return corpus

def TF_IDF(corpus):
    corpus = corpus.copy()
    total_number_of_documents = len(corpus)
    word_list = {}
    for key, document in corpus.items():
        for word, amount in document:
            if word not in word_list:
                word_list[word] = 1
            else:
                word_list[word] += 1
    
    for key, document in corpus.items():
        total_number_of_words = sum([amount for _, amount in document])
        new_document = []
        for word, amount in document:
            TF = amount / total_number_of_words
            IDF = np.log(total_number_of_documents / word_list[word])
            new_document.append((word, amount, TF, IDF, TF * IDF))
            
        corpus[key] = sorted(new_document, key = lambda x: x[4], reverse=True)
    
    return corpus

def create_wordclouds(corpus, width = 400, height = 300):

    def normalize(x, a, b):
            return (x-b)/(a-b) * 1000
    n = len(corpus)
    if n == 0:
        print("Corpus is empty.")
        return

    # --- Figure layout ---
    rows = 2  # number of columns in the grid
    cols = math.ceil(n / rows)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 8))
    axes = axes.flatten() if n > 1 else [axes]

    # --- Create word clouds for each community ---
    for i, (key, token_counts) in enumerate(corpus.items()):
        # Convert list of tuples into a frequency dictionary
        TF_IDF_list = [TF_IDF for _,_,_,_,TF_IDF in token_counts]
        max_, min_ = max(TF_IDF_list), min(TF_IDF_list)
        
        freq_dict = []
        for word, _, _, _, TF_IDF in token_counts:
            freq_dict.append((word, normalize(TF_IDF, max_, min_)))
        freq_dict = dict(freq_dict)
        if not freq_dict:
            axes[i].set_visible(False)
            continue

        wc = WordCloud(
            width=width,
            height=height,
            background_color='white',
            colormap='viridis',
            random_state=42
        ).generate_from_frequencies(freq_dict)

        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(key, fontsize=16)
        axes[i].axis('off')

    # --- Hide any unused subplots ---
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
G_undirected = G.to_undirected().copy()

#Resolution refer to how large the comunities should be. resolution < 1 will prefer larger comunitites and resolution > 1 prefer smaller communities
louvain_partition = nx.community.louvain_communities(G_undirected, resolution= 1.2, seed = 42) 
louvain_partition_corpus = create_corpus(wikipedia_data, louvain_partition)

corpus = TF_IDF(louvain_partition_corpus)
create_wordclouds(corpus)

##### Question: Use the matrix D (Lecture 7, part 2) to dicusss the difference between the word-clouds between genres and communities.

## Part 4: Sentiment of the artists and communities

The questions below are based on Lecture 8




##### Question: Calculate the sentiment of the band/artist pages (it is OK to work with the sub-network of artists-with-genre) and describe your findings using stats and visualization, inspired by the first exercise of week 8.

**Answer**:

##### Question: Discuss the sentiment of the communities. Do the findings using TF-IDF during Lecture 7 help you understand your results?

**Answer**:

### THE END

Contribution: We have each contributed equally to all parts of the assignment. If it is necessary to elaborate it can be somewhat fair to divide as follows: 
- Part 1: Both
- Part 2: Freja Tusindfryd Dollas 
- Part 3: Søren Stange
- Part 4:    